# LCEL 인터페이스

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `invoke`, `batch`, `stream` 동기 메서드의 차이를 설명하고 적절한 상황에 사용할 수 있어요
2. `ainvoke`, `abatch`, `astream` 비동기 메서드를 Jupyter 환경에서 실행할 수 있어요
3. `asyncio.gather`로 여러 LLM 호출을 병렬 실행해 성능을 높일 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 체인 구성 방법
- Python `async`/`await` 기초 (선택, 섹션 2에서 사용해요)

## 이전 노트북 연결

`02_Chain.ipynb`에서 체인을 구성하는 방법을 배웠어요. 이번에는 그 체인을 다양한 방식으로 실행하는 LCEL(LangChain Expression Language) 표준 인터페이스를 살펴볼게요.

아래 다이어그램은 LCEL 파이프라인에서 파이프 연산자가 데이터를 어떻게 전달하는지 보여줘요.

```mermaid
flowchart LR
    A["입력<br/>{question}"]:::input --> B["PromptTemplate<br/>|"]:::process
    B --> C["ChatOpenAI<br/>|"]:::process
    C --> D["StrOutputParser<br/>최종 출력"]:::output

    A2["invoke<br/>단일 실행"]:::process --> R1["결과 1개"]:::output
    A3["batch<br/>리스트 입력"]:::process --> R2["결과 리스트"]:::output
    A4["stream<br/>청크 스트리밍"]:::process --> R3["토큰 단위<br/>실시간 출력"]:::output
    A5["ainvoke / abatch<br/>/ astream"]:::process --> R4["비동기 결과"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

`Runnable(러너블)` 프로토콜은 LangChain의 대부분 컴포넌트에 구현되어 있어요. 표준 인터페이스를 통해 사용자 정의 체인을 정의하고 일관된 방식으로 호출할 수 있어요.

**동기 메서드**:
- `invoke`: 단일 입력을 처리하고 결과를 반환해요
- `batch`: 여러 입력을 일괄 처리해요
- `stream`: 응답 청크를 실시간으로 스트리밍해요

**비동기 메서드**:
- `ainvoke`, `abatch`, `astream`: 위 메서드의 비동기 버전이에요
- `astream_log`: 최종 응답뿐만 아니라 중간 단계도 스트리밍해요

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import asyncio
import time

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 기본 체인 구성: 프롬프트 → LLM → 문자열 추출
# 아래 섹션에서 invoke, batch, stream 등 다양한 방식으로 이 체인을 호출해요
prompt_template = PromptTemplate.from_template("{question}")
chain = prompt_template | model | StrOutputParser()

## 1. 동기 메서드 (Synchronous Methods)

동기 메서드는 각 작업이 완료될 때까지 다음 작업을 기다려요. 코드 흐름이 단순하고 예측하기 쉬워요.

세 가지 동기 메서드를 순서대로 살펴볼게요: `invoke` → `batch` → `stream`

### 1.1 invoke - 단일 입력 처리

`invoke`는 가장 기본적인 메서드예요. 단일 입력에 대해 체인을 실행하고 결과를 반환해요. 응답이 완전히 완료된 뒤에야 반환돼요.

In [2]:
# ---------------------------------------------------
# invoke(): 단일 입력 처리하고 완성된 응답 반환하기
# ---------------------------------------------------
# invoke()는 응답 전체가 완성될 때까지 블로킹 후 결과를 반환해요

result = chain.invoke({"question": "파이썬에서 함수를 정의하는 방법 알려줘."})
print(result)

파이썬에서 함수를 정의하는 방법은 `def` 키워드를 사용하는 것입니다. 함수 정의의 기본 문법은 다음과 같습니다:

```python
def 함수이름(매개변수1, 매개변수2, ...):
    """문서 문자열 (선택 사항)"""
    # 함수의 실행 코드
    return 반환값  # (선택 사항)
```

### 예시

1. **간단한 함수 정의 예시**:

```python
def 인사하다(이름):
    print(f"안녕하세요, {이름}님!")

인사하다("홍길동")  # 출력: 안녕하세요, 홍길동님!
```

2. **값을 반환하는 함수 예시**:

```python
def 덧셈(a, b):
    return a + b

결과 = 덧셈(3, 5)
print(결과)  # 출력: 8
```

3. **기본값을 가진 매개변수 사용**:

```python
def 인사하다(이름="손님"):
    print(f"안녕하세요, {이름}님!")

인사하다()  # 출력: 안녕하세요, 손님님!
인사하다("철수")  # 출력: 안녕하세요, 철수님!
```

4. **가변 인자를 사용하는 함수**:

```python
def 합계(*숫자들):
    return sum(숫자들)

print(합계(1, 2, 3))  # 출력: 6
print(합계(1, 2, 3, 4, 5))  # 출력: 15
```

이와 같이 파이썬에서 함수를 정의하고 사용할 수 있습니다. 필요에 따라 매개변수를 추가하거나, 기본값을 설정하거나, 가변 인자를 사용할 수 있습니다.


### 1.2 batch - 여러 입력 일괄 처리

`batch`는 입력 리스트를 받아 내부적으로 병렬 처리해 결과 리스트를 반환해요. 개별 `invoke()`를 여러 번 호출하는 것보다 빠를 수 있어요.

In [3]:
# ---------------------------------------------------
# batch(): 여러 입력을 리스트로 전달해 한 번에 처리하기
# ---------------------------------------------------
# ============================================================
# TODO: questions 리스트에 원하는 질문을 추가하고 batch()로 실행해봐요
# 힌트: chain.batch([{"question": "..."}, ...]) 형태로 호출해요
# 예상 결과: 각 질문에 대한 응답이 리스트로 반환돼요
# ============================================================

questions = [
    {"question": "파이썬이란 무엇인가요?"},
    {"question": "머신러닝이란 무엇인가요?"},
    {"question": "딥러닝이란 무엇인가요?"}
]

results = chain.batch(questions)
print(results)

['파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 발표되었습니다. 파이썬은 읽기 쉽고, 문법이 간결하여 초보자에게 접근하기 쉬운 언어로 알려져 있습니다. 또한, 강력한 라이브러리와 프레임워크를 지원하여 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트 등 다양한 분야에서 널리 사용됩니다.\n\n파이썬의 주요 특징은 다음과 같습니다:\n\n1. **읽기 쉬운 문법**: 파이썬의 문법은 자연어에 가까워서 프로그램을 이해하기 쉽습니다.\n2. **다양한 라이브러리**: 데이터 과학, 웹 개발, 머신러닝 등 다양한 분야를 위한 풍부한 라이브러리가 제공됩니다.\n3. **플랫폼 독립성**: 대부분의 운영체제에서 실행할 수 있습니다.\n4. **대화형 프로그래밍**: REPL(Read-Eval-Print Loop)을 통해 코드를 즉시 실행하고 결과를 확인할 수 있는 기능을 제공합니다.\n5. **객체 지향 프로그래밍**: 객체와 클래스 개념을 지원하여 재사용성이 높고 구조화된 코드를 작성할 수 있도록 합니다.\n\n이러한 이유들로 인해 파이썬은 인기 있는 프로그래밍 언어 중 하나로 자리잡고 있으며, 교육 및 산업 전반에서 많이 활용되고 있습니다.', '머신러닝(Machine Learning)은 인공지능(AI)의 한 분야로, 컴퓨터가 명시적인 프로그래밍 없이도 데이터를 통해 학습하고 개선할 수 있는 기술을 의미합니다. 머신러닝은 알고리즘과 통계 모델을 사용하여 데이터를 분석하고, 패턴을 인식하며, 예측을 수행하는 데 중점을 둡니다.\n\n머신러닝은 크게 세 가지 주요 유형으로 분류됩니다:\n\n1. **지도 학습(Supervised Learning)**: 입력 데이터와 그에 대응하는 출력 데이터(레이블)가 주어졌을 때, 모델이 입력과 출력 간의 관계를 학습하도록 하는 방법입니다. 예를 들어, 스팸 이메일을 분류하는 모델이 있을 때, 라벨이 있는 이메일 데이터를 사용하여 학습합니다.\n\n2

### 1.3 stream - 스트리밍 응답

`stream`은 LLM이 토큰을 생성할 때마다 즉시 청크(chunk)를 전달해요. 긴 응답을 처리할 때 사용자가 첫 결과를 더 빨리 볼 수 있어요.

> **실무 팁**: 챗봇 UI처럼 응답을 실시간으로 표시해야 하는 경우 `stream`을 사용해요. 배치 처리나 응답 전체를 저장해야 할 때는 `invoke`가 더 적합해요.

In [ ]:
# ---------------------------------------------------
# stream(): 토큰이 생성되는 즉시 화면에 출력하기
# ---------------------------------------------------
# ChatGPT 웹 화면에서 글자가 한 글자씩 나타나는 것과 같은 원리예요
# 응답 완성을 기다리지 않으므로 사용자 체감 속도가 향상돼요


### 1.4 stream - 청크 수집해 전체 메시지 구성

스트리밍 청크를 변수에 누적하면 전체 응답도 함께 얻을 수 있어요. 실시간 출력과 전체 저장을 동시에 처리하는 패턴이에요.

In [ ]:
# ---------------------------------------------------
# 스트리밍 청크를 누적해 전체 응답 함께 구성하기
# ---------------------------------------------------
# 실시간 출력과 전체 텍스트 저장을 동시에 처리하는 패턴이에요

stream_result = chain.stream({"question": "파이썬의 특징 세 가지를 정리하세요."})
print(stream_result)

# chunk : 스트림을 통과한 한 개의 토큰
for chunk in stream_result:
    print(chunk, end="", flush=True)

<generator object RunnableSequence.stream at 0x137a1e4d0>
파이썬의 주요 특징 세 가지는 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**:
   파이썬은 코드의 가독성을 중요시합니다. 배우기 쉽고 문법이 간결하여 초보자도 쉽게 이해하고 사용할 수 있습니다. 들여쓰기를 사용하여 블록을 구분하는 방식은 코드 구조를 명확하게 해줍니다.

2. **다양한 라이브러리와 패키지**:
   파이썬은 방대한 표준 라이브러리와 서드파티 라이브러리를 제공합니다. 이를 통해 다양한 분야(데이터 과학, 인공지능, 웹 개발 등)에서 필요한 기능을 쉽게 구현할 수 있습니다. `NumPy`, `Pandas`, `TensorFlow`, `Django` 등 많은 인기 있는 라이브러리가 존재합니다.

3. **플ATFORM 독립성**:
   파이썬은 다양한 운영 체제(Windows, macOS, Linux 등)에서 실행할 수 있는 플랫폼 독립적인 언어입니다. 한 번 작성한 코드는 운영 체제에 상관없이 동일하게 실행될 수 있으며, 이는 개발자의 생산성을 증가시킵니다. 

이 외에도 객체 지향 프로그래밍, 동적 타이핑, 인터프리터 언어 등 여러 장점이 있는 언어입니다.

## 2. 비동기 메서드 (Asynchronous Methods)

앞서 동기 메서드를 살펴봤어요. 이제 비동기(async) 메서드를 알아볼게요.

LLM API 호출 시간의 대부분은 **네트워크 응답 대기**예요. 동기 방식에서는 대기하는 동안 아무것도 못 하지만, 비동기 방식에서는 그 시간에 다른 API 호출을 시작할 수 있어요.

| 방식 | 동작 |
|------|------|
| 동기 `invoke` 3회 | A 완료 대기 → B 완료 대기 → C 완료 대기 (총 시간 = A + B + C) |
| 비동기 `ainvoke` 3회 (`asyncio.gather`) | A, B, C 동시 시작 → 가장 느린 것만큼 대기 (총 시간 ≈ max(A, B, C)) |

> **실무 팁**: FastAPI 같은 웹 서버 환경에서는 비동기가 필수예요. 한 사용자의 LLM 응답을 기다리는 동안 다른 사용자 요청도 처리할 수 있기 때문이에요.

### 2.1 ainvoke - 비동기 단일 입력 처리

`ainvoke`는 `invoke`의 비동기 버전이에요. Jupyter 노트북은 이미 실행 중인 이벤트 루프가 있어서 `asyncio.run()` 대신 `await`를 셀에서 직접 사용할 수 있어요.

> **주의**: 일반 Python 스크립트에서는 `asyncio.run(chain.ainvoke(...))` 형태로 실행해야 해요. Jupyter에서만 `await`를 최상위 레벨에서 사용할 수 있어요.

In [8]:
# ---------------------------------------------------
# ainvoke(): 비동기 단일 호출 (Jupyter에서 await 직접 사용)
# ---------------------------------------------------
# Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 await를 셀 최상위에서 사용 가능
# 일반 Python 스크립트에서는 asyncio.run(chain.ainvoke(...)) 형태로 실행해야 해요

result = await chain.ainvoke({"question": "비동기 프로그래밍이란 무엇인가요?"})
print(result)

비동기 프로그래밍(Asynchronous Programming)은 프로그램의 실행 흐름을 효율적으로 관리하기 위해, 특정 작업이 완료될 때까지 기다리지 않고 다른 작업을 처리할 수 있도록 하는 프로그래밍 패러다임입니다. 즉, 비동기 프로그래밍에서는 특정 작업이 완료되는 것을 기다리지 않고 나머지 작업을 동시에 수행할 수 있습니다.

비동기 프로그래밍의 주요 개념은 다음과 같습니다:

1. **콜백(Callback)**: 비동기 작업이 완료되었을 때 호출되는 함수입니다. 작업이 완료되면 이 콜백 함수가 실행됩니다.

2. **프라미스(Promise)**: 비동기 작업의 결과를 나타내는 객체입니다. 프라미스는 세 가지 상태(대기 중, 이행됨, 거부됨)를 가지며, 작업이 성공적으로 완료되면 이행됨 상태로 전환되고, 실패하면 거부됨 상태로 전환됩니다.

3. **async/await**: 비동기 코드를 작성할 때 사용하는 문법입니다. `async` 키워드로 정의된 함수는 항상 프라미스를 반환하며, `await` 키워드는 프라미스가 이행될 때까지 기다립니다. 이를 통해 비동기 처리를 보다 직관적으로 작성할 수 있습니다.

비동기 프로그래밍의 장점은 다음과 같습니다:

- **효율적인 자원 사용**: 비동기 처리를 통해 프로그램은 블로킹 없이 다른 작업을 수행할 수 있어, CPU와 메모리 자원을 효율적으로 사용할 수 있습니다.

- **응답성 향상**: 사용자 인터페이스(UI)를 가진 프로그램에서는 비동기 프로그래밍을 통해 사용자 인터페이스가 차단되지 않도록 하여, 더욱 매끄러운 사용자 경험을 제공합니다.

- **복잡한 작업 처리**: 네트워크 요청, 파일 I/O 등 시간이 걸리는 작업을 비동기적으로 처리하여, 전체 프로그램의 성능을 향상시킬 수 있습니다.

비동기 프로그래밍은 JavaScript, Python, C# 등 다양한 프로그래밍 언어에서 지원되며, 현대 소프트웨어 개발에서 중요한 패턴 중 하나로 자리 잡고 있습니다.


### 2.2 abatch - 비동기 일괄 처리

`abatch`는 `batch`의 비동기 버전이에요. 내부적으로 `asyncio.gather`를 사용해 여러 입력을 동시에 처리해요.

In [10]:
# ---------------------------------------------------
# abatch(): 비동기 일괄 처리 (내부적으로 asyncio.gather 사용)
# ---------------------------------------------------
questions = [
    {"question": "파이썬이란 무엇인가요?"},
    {"question": "머신러닝이란 무엇인가요?"},
    {"question": "딥러닝이란 무엇인가요?"}
]

results = await chain.abatch(questions)
print(results)

['파이썬(Python)은 고급 프로그래밍 언어로, 코드의 가독성이 뛰어나고 간결한 문법을 가지고 있어 배우기 쉽습니다. 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었으며, 현재는 다양한 분야에서 널리 사용되고 있습니다.\n\n파이썬의 주요 특징은 다음과 같습니다:\n\n1. **가독성**: 파이썬 코드는 쉽게 읽고 이해할 수 있어 유지보수가 용이합니다.\n2. **비교적 쉬운 문법**: 다른 프로그래밍 언어에 비해 문법이 간단하여 초보자도 쉽게 배울 수 있습니다.\n3. **다양한 라이브러리**: 데이터 과학, 웹 개발, 인공지능, 머신러닝 등 다양한 분야에 맞는 풍부한 라이브러리와 프레임워크가 제공됩니다. 예를 들어, NumPy, Pandas, TensorFlow, Flask, Django 등이 있습니다.\n4. **플랫폼 독립적**: 파이썬은 윈도우, MacOS, 리눅스 등 다양한 운영체제에서 사용 가능합니다.\n5. **대화형 프로그래밍**: 파이썬은 대화형 모드에서 직접 코드를 실행할 수 있어 실험과 테스트가 용이합니다.\n\n이러한 특징들로 인해 파이썬은 초보자뿐만 아니라 전문가들 사이에서도 인기가 높은 언어입니다. 웹 애플리케이션, 데이터 분석, 인공지능, 자동화 스크립트 등 여러 분야에서 활용되고 있습니다.', '머신러닝(Machine Learning)은 인공지능(AI)의 한 분야로, 데이터와 경험을 통해 컴퓨터가 스스로 학습하고 예측할 수 있도록 하는 알고리즘과 기술을 연구하는 학문입니다. 머신러닝의 주요 목표는 명시적으로 프로그래밍하지 않고도 컴퓨터가 특정 작업을 수행할 수 있게 하는 것입니다.\n\n머신러닝은 크게 세 가지 유형으로 나눌 수 있습니다:\n\n1. **지도 학습(Supervised Learning)**: 입력 데이터와 그에 대한 정답(label)이 주어지는 경우, 모델이 이 데이터를 학습하여 새로운 입력에 대한 예측을 수행합니다. 예를 들어, 이메일이 스팸인지 아닌지를 분류하는 작업이 있습니다.

/var/folders/h6/vm46m4nx1xz4c_2phk_wz39h0000gn/T/ipykernel_98697/752293377.py:10: RuntimeWarning: coroutine 'RunnableSequence.abatch' was never awaited
  results = await chain.abatch(questions)


### 2.3 astream - 비동기 스트리밍

`astream`은 `stream`의 비동기 버전이에요. `async for`로 청크를 하나씩 받아요. 이벤트 루프가 블로킹되지 않아 웹소켓으로 실시간 전송하면서 다른 작업을 동시에 수행할 수 있어요.

In [13]:
# ---------------------------------------------------
# astream(): 비동기 스트리밍 (async for로 청크 수신)
# ---------------------------------------------------
# astream()은 이벤트 루프를 블로킹하지 않아요
# 웹소켓으로 사용자에게 실시간
stream_result = chain.astream({"question": "파이썬의 특징 세 가지를 정리하세요."})
print(stream_result)

# chunk : 스트림을 통과한 한 개의 토큰
async for chunk in stream_result:
    print(chunk, end="", flush=True)

<async_generator object RunnableSequence.astream at 0x137aeb440>
파이썬의 특징 세 가지는 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 직관적이고 명료하게 작성될 수 있도록 설계되었습니다. 이를 통해 개발자는 코드의 의도를 쉽게 이해하고, 유지 보수 및 협업이 용이합니다.

2. **강력한 표준 라이브러리**: 파이썬은 다양한 기능을 제공하는 풍부한 표준 라이브러리를 갖추고 있습니다. 이를 통해 파일 I/O, 웹 프로그래밍, 데이터 처리 등 다양한 작업을 쉽게 수행할 수 있으며, 외부 라이브러리도 널리 사용됩니다.

3. **다양한 플랫폼 지원**: 파이썬은 Windows, macOS, Linux 등 여러 운영 체제에서 실행될 수 있으며, 이식성이 뛰어나기 때문에 다양한 환경에서 사용할 수 있습니다. 또한, 다양한 분야(웹 개발, 데이터 분석, 인공지능 등)에서 널리 사용되고 있습니다.

## 3. 동기 vs 비동기 성능 비교

비동기 메서드를 살펴봤어요. 이제 실제로 시간을 측정해 동기와 비동기 방식의 차이를 직접 확인해볼게요.

> **참고**: `batch()`와 `abatch()`는 모두 내부적으로 최적화가 되어 있어 성능 차이가 크지 않을 수 있어요. 병렬 처리의 이점을 명확히 보려면 개별 `invoke` 순차 호출과 개별 `ainvoke` 병렬 호출을 비교해야 해요.

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| 메서드 | 설명 | 사용 상황 |
|--------|------|-----------|
| `invoke` | 단일 입력, 응답 완료 후 반환 | 단순 단일 호출 |
| `batch` | 여러 입력 일괄 처리 | 다수 입력을 동시에 처리할 때 |
| `stream` | 토큰 단위 실시간 출력 | 챗봇 UI, 긴 응답 실시간 표시 |
| `ainvoke` | 비동기 단일 호출 | 웹 서버, 비동기 환경 |
| `abatch` | 비동기 일괄 처리 | 비동기 환경에서 다수 입력 |
| `astream` | 비동기 스트리밍 | 웹소켓 실시간 전송 |

## 다음 노트북 예고

다음 `04_Runnable_Basic.ipynb`에서는 `RunnablePassthrough`, `RunnableParallel`, `RunnableLambda` 세 가지 핵심 Runnable을 조합해 복잡한 파이프라인을 구성하는 방법을 배울 거예요.

In [14]:
# ---------------------------------------------------
# 동기 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 코드를 실행하고 순차 invoke vs 병렬 ainvoke 시간 차이를 확인해요
# 힌트: asyncio.gather()로 여러 코루틴을 동시에 실행해요
# 예상 결과: 비동기 병렬 방식이 순차 방식보다 약 2~3배 빠를 거예요
# ============================================================

questions = [
    {"question": "파이썬이란?"},
    {"question": "머신러닝이란?"},
    {"question": "딥러닝이란?"}
]

print("방법 1: batch() vs abatch() 비교")
print("-" * 50)

# 동기 batch — 내부적으로 ThreadPoolExecutor로 병렬 처리
start_time = time.time()
sync_results = chain.batch(questions)
sync_time = time.time() - start_time
print(f"동기 방식 (batch): {sync_time:.2f}초")

# 비동기 abatch — 내부적으로 asyncio.gather로 병렬 처리
start_time = time.time()
async_results = await chain.abatch(questions)
async_time = time.time() - start_time
print(f"비동기 방식 (abatch): {async_time:.2f}초")

if sync_time > async_time:
    print(f"성능 향상: {sync_time/async_time:.2f}배")
else:
    print(f"성능 차이: {async_time/sync_time:.2f}배 느림 (내부 최적화로 인해 차이가 작을 수 있음)")

# 개별 invoke 순차 호출 vs 개별 ainvoke 병렬 호출 — 차이가 더 명확하게 나타나요
print("\n" + "=" * 50)
print("방법 2: 개별 invoke vs 개별 ainvoke 비교 (실제 병렬 처리)")
print("-" * 50)

# 순차 실행: 각 호출이 끝나야 다음 호출 시작
def sync_individual_invoke():
    start_time = time.time()
    results = []
    for q in questions:
        result = chain.invoke(q)
        results.append(result)
    return time.time() - start_time, results

# 병렬 실행: asyncio.gather로 3개 요청을 동시에 발사
# 총 시간 ≈ 가장 느린 1개 요청의 시간 (세 요청의 네트워크 대기가 겹침)
async def async_individual_invoke():
    start_time = time.time()
    tasks = [chain.ainvoke(q) for q in questions]  # 코루틴 리스트 생성 (아직 실행 X)
    results = await asyncio.gather(*tasks)           # 동시에 실행 시작
    return time.time() - start_time, results

sync_ind_time, sync_ind_results = sync_individual_invoke()
print(f"동기 방식 (개별 invoke, 순차): {sync_ind_time:.2f}초")

async_ind_time, async_ind_results = await async_individual_invoke()
print(f"비동기 방식 (개별 ainvoke, 병렬): {async_ind_time:.2f}초")

if sync_ind_time > async_ind_time:
    print(f"성능 향상: {sync_ind_time/async_ind_time:.2f}배")
else:
    print(f"성능 차이: {async_ind_time/sync_ind_time:.2f}배")

print("\n" + "=" * 50)
print("결론:")
print("- batch()와 abatch()는 모두 내부적으로 최적화되어 있어 성능 차이가 작을 수 있습니다")
print("- 실제 병렬 처리의 이점을 보려면 개별 invoke를 순차 호출 vs 개별 ainvoke를 병렬 호출로 비교해야 합니다")
print("- 요청 수가 많을수록 비동기 방식의 이점이 더 명확해집니다")

방법 1: batch() vs abatch() 비교
--------------------------------------------------
동기 방식 (batch): 5.92초
비동기 방식 (abatch): 6.44초
성능 차이: 1.09배 느림 (내부 최적화로 인해 차이가 작을 수 있음)

방법 2: 개별 invoke vs 개별 ainvoke 비교 (실제 병렬 처리)
--------------------------------------------------
동기 방식 (개별 invoke, 순차): 17.71초
비동기 방식 (개별 ainvoke, 병렬): 8.19초
성능 향상: 2.16배

결론:
- batch()와 abatch()는 모두 내부적으로 최적화되어 있어 성능 차이가 작을 수 있습니다
- 실제 병렬 처리의 이점을 보려면 개별 invoke를 순차 호출 vs 개별 ainvoke를 병렬 호출로 비교해야 합니다
- 요청 수가 많을수록 비동기 방식의 이점이 더 명확해집니다
